## [Advent Of Code 2015/15](https://adventofcode.com/2015/day/15)

Launch this notebook [here](https://datalore.jetbrains.com/view/notebook/5rVU0OC2dt9WbbfSPn1MhT)

### Part1

Looks like linear programming problem.

Goal is to [maximize product of linear functions](https://math.stackexchange.com/questions/324568/maximize-the-product-of-linear-functions)

NB: Solution should be integer (sorry, `scipy.optimize`, it is not your time yet).


In [8]:
import sys
!{sys.executable} -m pip install --upgrade pip > /dev/null
!{sys.executable} -m pip install numpy pulp > /dev/null

import numpy as np
from pulp import *
import re

input_file = "15_test.txt"

matchRe = re.compile(r'^(?P<name>\w+): '
                     r'capacity (?P<capacity>-?\d+), '
                     r'durability (?P<durability>-?\d+), '
                     r'flavor (?P<flavor>-?\d+), '
                     r'texture (?P<texture>-?\d+), '
                     r'calories (?P<calories>-?\d+)$')


Ingredients = []
Components = []


def list_props(str_line: str) -> list:
    m = matchRe.match(str_line)
    Ingredients.append(m.group('name'))
    return list(map(int, m.groups()[1::]))


with open(input_file) as f:
    # noinspection PyRedeclaration
    Components = list(map(list_props, f))

# print(components)

# strip calories column
Components = np.delete(Components, 4, 1)
print(Ingredients)
print(Components)


['Butterscotch', 'Cinnamon']
[[-1 -2  6  3]
 [ 2  3 -2 -1]]


In [13]:
def calc_score(weights: list) -> int:
    if len(Components) != len(weights):
        return 0

    # Multiply components matrix × weights matrix, e.g. calc_score([44, 56]):
    # capacity of 44*-1 + 56*2 = 68
    # durability of 44*-2 + 56*3 = 80
    # flavor of 44*6 + 56*-2 = 152
    # texture of 44*3 + 56*-1 = 76
    properties = np.matmul(np.transpose(Components), weights)

    # If any properties had produced a negative total,
    # it would have instead become zero,
    # causing the whole score to multiply to zero.
    for number in properties:
        if number < 0:
            return 0

    # Get product of the resulting matrix.
    # e.g.: 68 * 80 * 152 * 76
    product = np.prod(properties)
    return product


# weights: list of decision variables
def sum_weights(weights: list):
    return np.sum(weights)


Weights = [44, 56]

print(sum_weights(Weights))
total_score = calc_score(Weights)
print(total_score)

100
62842880


In [18]:
cmp_count = len(Components)

problem = LpProblem("Cookie_Scoring", LpMaximize)

# Define Decision Variables
variableList = [
    pulp.LpVariable('{}'.format(ingredient), cat='Integer', lowBound=1, upBound=99) for ingredient in Ingredients
]

# variableDict = LpVariable.dicts(
#     'ingredient_weight_',
#     Ingredients,
#     lowBound=1,
#     upBound=99,
#     cat='Integer'
# )

constraint = LpConstraint(
    lpSum(variableList) - 100,
    LpConstraintEQ,
    name='Sum_of_ingredient_weights'
)


0


TypeError: lpDot() missing 1 required positional argument: 'v2'